# Wind statistical baselines (Darts)




In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent

ROOT=REPO_ROOT/"wind"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path:
    sys.path.insert(0,str(ROOT/"src"))

from darts import TimeSeries, concatenate
from darts.models import NaiveSeasonal, AutoARIMA, AutoTheta, AutoETS
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from preprocessing import load_wind, describe_wind, NWP_FEATURES, TEST_START, SITES
from src.metrics import print_metrics
from power_conversion import convert_wind_result_to_power

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
NUM_SAMPLES=100

LOAD_EXISTING=False
SAVE_MODELS=True

MODEL_DIR=ROOT/"models"/"darts"/"statistical"
OUTPUT_DIR=ROOT/"outputs"/"predictions"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Flat data layout: wind/data/hub-height wind.csv and wind/data/nwp_data.csv
data=load_wind(ROOT/"data")
print(describe_wind(data))


## Build per-site target and NWP series


In [ ]:
series=[]
future_covariates=[]

for item_id,site in enumerate(SITES,start=1):
    site_df=(
        data.full[data.full["item_id"]==item_id]
        .sort_values("timestamp")
        .copy()
    )

    ts=TimeSeries.from_dataframe(
        site_df,
        time_col="timestamp",
        value_cols="target",
        freq="h",
    ).astype(np.float32)

    cov=TimeSeries.from_dataframe(
        site_df,
        time_col="timestamp",
        value_cols=NWP_FEATURES,
        freq="h",
    ).astype(np.float32)

    series.append(ts)
    future_covariates.append(cov)

test_start=pd.Timestamp(TEST_START)


train_series=[]
train_covariates=[]

for ts,cov in zip(series,future_covariates):
    train,_=ts.split_before(test_start)
    cov_train,_=cov.split_before(test_start)
    train_series.append(train)
    train_covariates.append(cov_train)

for site,ts,cov in zip(SITES,series,future_covariates):
    print(
        site,
        "| target:",ts.start_time(),"->",ts.end_time(),
        "| future covariates:",cov.start_time(),"->",cov.end_time(),
    )
    assert cov.end_time()>=ts.end_time()


## Helpers


In [ ]:
MODEL_CLASSES={
    "auto_arima":AutoARIMA,
    "auto_theta":AutoTheta,
    "auto_ets":AutoETS,
}

def build_stat(name):
    if name=="auto_arima":
        return AutoARIMA(
            seasonal=True,
            season_length=24,
            stepwise=True,
            trace=True,
            d=1,
            D=1,
            max_p=1,
            max_q=1,
            max_P=1,
            max_Q=1,
        )
    if name=="auto_theta":
        return AutoTheta(season_length=24)
    if name=="auto_ets":
        return AutoETS(season_length=24)
    raise ValueError(name)

def forecast_frame(series_list,forecast_list,probabilistic=True):
    rows=[]

    for item_id,(ts,fc) in enumerate(
        zip(series_list,forecast_list),
        start=1,
    ):
        gt=ts.slice_intersect(fc)
        fc=fc.slice_intersect(gt)

        out=pd.DataFrame({
            "item_id":item_id,
            "timestamp":gt.time_index,
            "target":gt.values().flatten(),
        })

        if probabilistic:
            for q in OUTPUT_QUANTILES:
                out[str(q)]=fc.quantile(q).values().flatten()
            out["prediction"]=out["0.5"]
        else:
            out["prediction"]=fc.values().flatten()
            for q in OUTPUT_QUANTILES:
                out[str(q)]=np.nan

        rows.append(out)

    return (
        pd.concat(rows,ignore_index=True)
        .sort_values(["item_id","timestamp"])
        .reset_index(drop=True)
    )


## Seasonal Naive


In [ ]:
forecasts=[]

for site,ts,train in zip(SITES,series,train_series):
    model=NaiveSeasonal(K=24)
    model.fit(train)

    bt=model.historical_forecasts(
        series=ts,
        start=test_start,
        forecast_horizon=24,
        stride=24,
        retrain=True,
        last_points_only=False,
        verbose=True,
    )
    forecasts.append(concatenate(bt))

result=forecast_frame(
    series,
    forecasts,
    probabilistic=False,
)

result.to_csv(
    OUTPUT_DIR/"seasonal_naive.csv",
    index=False,
)

print("Seasonal Naive")
print(f"MAE:  {mean_absolute_error(result.target,result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(result.target,result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")

power_result=convert_wind_result_to_power(result)
power_result.to_csv(
    OUTPUT_DIR/"seasonal_naive_power.csv",
    index=False,
)

print("Seasonal Naive [wind power]")
print(f"MAE:  {mean_absolute_error(power_result.target,power_result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(power_result.target,power_result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")


## Auto-ARIMA, Auto-Theta, Auto-ETS


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    forecasts=[]

    for site,ts,train,cov,cov_train in zip(
        SITES,
        series,
        train_series,
        future_covariates,
        train_covariates,
    ):
        path=MODEL_DIR/f"{name}_{site}.darts"

        if LOAD_EXISTING and path.exists():
            model=MODEL_CLASSES[name].load(str(path))
        else:
            model=build_stat(name)
            model.fit(
                series=train,
                future_covariates=cov_train,
            )
            if SAVE_MODELS:
                model.save(str(path))


        bt=model.historical_forecasts(
            series=ts,
            future_covariates=cov,
            start=test_start,
            forecast_horizon=24,
            stride=24,
            num_samples=NUM_SAMPLES,
            retrain=True,
            last_points_only=False,
            verbose=True,
        )

        forecasts.append(concatenate(bt))

    result=forecast_frame(
        series,
        forecasts,
        probabilistic=True,
    )

    result.to_csv(
        OUTPUT_DIR/f"{name}.csv",
        index=False,
    )
    print_metrics(
        result,
        BASE_QUANTILES,
        label=name,
    )

    power_result=convert_wind_result_to_power(result)
    power_result.to_csv(
        OUTPUT_DIR/f"{name}_power.csv",
        index=False,
    )
    print_metrics(
        power_result,
        BASE_QUANTILES,
        label=f"{name} [wind power]",
    )
